In [38]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

if os.environ["OPENAI_API_KEY"] is None:
    raise ValueError("OPENAI_API_KEY environment variable not set.")
else:
    print("OPENAI_API_KEY environment variable is set.")


OPENAI_API_KEY environment variable is set.


In [5]:
# Verify installation
import autogen_agentchat
import autogen_ext
print(f"autogen-agentchat : {autogen_agentchat.__version__}")
print(f"autogen-ext       : {autogen_ext.__version__}")
print("✅ All packages installed correctly")

autogen-agentchat : 0.7.5
autogen-ext       : 0.7.1
✅ All packages installed correctly


In [41]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Create an agent that uses the OpenAI GPT-4o model.
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    # api_key="YOUR_API_KEY",
)

## Graph flow

-> DiGraph helps you control the execution between agents
-> Sequential , Parallel, Conditiondal, looping

Patterns:
1. Sequential Flow  ( Writer -> Reviwer)
2. parallel flow (writer -> Editor1.    -> Review )
                         -> Editor2 
3. Conditional flow


There are cases multiple edges points to single node, these will be handled by activation groups 

- all(where it waits for both the edges to complete the task)
- any ( if any one of edges completes it start executing it)

## Conditional Branch

In [12]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import GraphFlow, DiGraphBuilder
from autogen_agentchat.ui import Console


classifier = AssistantAgent(name="Classifier", model_client=model_client,system_message="""You are a helpful assistant that classifies the support ticket.
                            Respond with only one of the following categories: "BILLING", "TECHNICAL", "GENERAL".""")


billing_agent = AssistantAgent(name="BillingAgent", model_client=model_client,system_message="""You are a helpful assistant that responds to billing-related support
                                tickets. Provide clear and concise answers to the user's billing questions. END with RESOLVED""")

tech_agent = AssistantAgent(name="TechAgent", model_client=model_client,system_message="""You are a helpful assistant that responds to technical support-related support
                                tickets. Provide clear and concise answers to the user's technical questions. END with RESOLVED""")

general_inquiry_agent = AssistantAgent(name="GeneralInquiryAgent", model_client=model_client,system_message="""You are a helpful assistant that responds to general inquiry-related support
                                tickets. Provide clear and concise answers to the user's general questions. END with RESOLVED""")


def is_billing(msg:str)->bool:
    return "BILLING" in msg.content.upper()

def is_technical(msg:str)->bool:
    return "TECHNICAL" in msg.content.upper()

def is_general(msg:str)->bool:
    return "GENERAL" in msg.content.upper()


builder = DiGraphBuilder()
builder.add_node(classifier)
builder.add_node(billing_agent)
builder.add_node(tech_agent)
builder.add_node(general_inquiry_agent)

builder.add_edge(classifier, billing_agent, condition=is_billing)
builder.add_edge(classifier, tech_agent, condition=is_technical)
builder.add_edge(classifier, general_inquiry_agent, condition=is_general)

graph = builder.build()
team = GraphFlow([classifier, billing_agent, tech_agent, general_inquiry_agent], graph)


In [13]:
result = await Console(team.run_stream(task="""Support ticket from: dev@acme.com
Subject: API returning 429 errors constantly
Message: Our integration has been failing for 2 hours. We're getting
HTTP 429 Too Many Requests on every call even though we're well within
our supposed rate limits. This is blocking our production deployment."""))

print(f"STOP REASON:{result.stop_reason}")
print(f"TOTAL MESSAGES:len({result.messages}")

---------- TextMessage (user) ----------
Support ticket from: dev@acme.com
Subject: API returning 429 errors constantly
Message: Our integration has been failing for 2 hours. We're getting
HTTP 429 Too Many Requests on every call even though we're well within
our supposed rate limits. This is blocking our production deployment.
---------- TextMessage (Classifier) ----------
TECHNICAL
---------- TextMessage (TechAgent) ----------
HTTP 429 errors indicate that you've exceeded the rate limits imposed by the API. Even if you believe you are within your limits, there are a few steps you can take to troubleshoot this issue:

1. **Check Rate Limits**: Confirm the current rate limits set for your API access. Sometimes, these limits can vary based on the endpoint or the tier of service you are using.

2. **Aggregation of Requests**: If your application is making requests rapidly (even if they fall within the average limit), ensure that you don't exceed the burst capacity. Try slowing down your 

In [14]:
result = await Console(team.run_stream(task="""Support ticket from: finance@corp.com
Subject: Charged twice this month
Message: We were charged $299 on March 1st and again on March 15th.
We only have one subscription. Please refund the duplicate charge."""))

print(f"STOP REASON:{result.stop_reason}")
print(f"TOTAL MESSAGES:len({result.messages}")

---------- TextMessage (user) ----------
Support ticket from: finance@corp.com
Subject: Charged twice this month
Message: We were charged $299 on March 1st and again on March 15th.
We only have one subscription. Please refund the duplicate charge.
---------- TextMessage (Classifier) ----------
BILLING
---------- TextMessage (BillingAgent) ----------
I understand your concern regarding the duplicate charge on your account for the subscription. We will certainly assist you with this issue. 

To process your request for a refund, please provide the following information:
1. The last four digits of the credit card used for the subscription.
2. The confirmation number or email for the transactions, if available.

Once I have this information, I can help expedite the refund for the duplicate charge. Thank you for your patience. RESOLVED
STOP REASON:Digraph execution is complete
TOTAL MESSAGES:len([TextMessage(id='4cfc9229-c10d-48df-8cd8-ad83966302b1', source='user', models_usage=None, metada

## SWARM

Agent A   Agent B   Agent C

Agent D   Agent E   Agent F 


handoff Message  -> No Central orchestrator , each agent makes its own decision about who to pass the next baton.
All agents will share the common context / hsitory.


1. Customer Support (human in loop) : User   travel agent.  flight refunder 

2. 

## https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/swarm.html

In [30]:
from typing import Any, Dict, List

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.messages import HandoffMessage
from autogen_agentchat.teams import Swarm
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient


In [31]:
def refund_flight(flight_id: str) -> str:
    """Refund a flight"""
    return f"Flight {flight_id} refunded"


In [32]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    # api_key="YOUR_API_KEY",
)

travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""You are a travel agent.
    The flights_refunder is in charge of refunding flights.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    Use TERMINATE when the travel planning is complete.""",
)

flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent", "user"],
    tools=[refund_flight],
    system_message="""You are an agent specialized in refunding flights.
    You only need flight reference numbers to refund a flight.
    You have the ability to refund a flight using the refund_flight tool.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    When the transaction is complete, handoff to the travel agent to finalize.""",
)


In [33]:
termination = HandoffTermination(target="user") | TextMentionTermination("TERMINATE")
team = Swarm([travel_agent, flights_refunder], termination_condition=termination)


In [ ]:
task = "I need to refund my flight."


async def run_team_stream() -> None:
    task_result = await Console(team.run_stream(task=task))
    last_message = task_result.messages[-1]
    print(f"Last message: {last_message}")

    while isinstance(last_message, HandoffMessage) and last_message.target == "user":
        user_message = input("User: ")

        task_result = await Console(
            team.run_stream(task=HandoffMessage(source="user", target=last_message.source, content=user_message))
        )
        last_message = task_result.messages[-1]


# Use asyncio.run(...) if you are running this in a script.
await run_team_stream()
#await model_client.close()


---------- TextMessage (user) ----------
I need to refund my flight.
---------- ToolCallRequestEvent (travel_agent) ----------
[FunctionCall(id='call_SFdXTaRIO09LPsN05RKBId3D', arguments='{}', name='transfer_to_user')]
---------- ToolCallExecutionEvent (travel_agent) ----------
[FunctionExecutionResult(content='Transferred to user, adopting the role of user immediately.', name='transfer_to_user', call_id='call_SFdXTaRIO09LPsN05RKBId3D', is_error=False)]
---------- HandoffMessage (travel_agent) ----------
Transferred to user, adopting the role of user immediately.
Last message: id='c63585ef-e2f4-48df-ab49-dbeba1cf1973' source='travel_agent' models_usage=None metadata={} created_at=datetime.datetime(2026, 4, 11, 4, 49, 53, 667464, tzinfo=datetime.timezone.utc) content='Transferred to user, adopting the role of user immediately.' target='user' context=[] type='HandoffMessage'
---------- HandoffMessage (user) ----------
Refund the flight ticket
---------- ToolCallRequestEvent (travel_agent

## Assistant Agent, Code Executor Agent, UserProxy Agent, Society of Mind Agent 


Society of Mind agent


Agent 1,  agent 2 and agent 3


agent 1 and agent 2 -> team (inner team)

agent 3 



Agent 1 -> writer
Agent 2 -> editor


Agent 1 ang Agent 2 -> Any team ( round robin, selector group, graph or swarm) -- SOM team

Agent 3 -> Reviewer

Any team( round robin, selector group, graph or swarm) -> (SOM team and Agent 3)




In [42]:
import asyncio
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent, SocietyOfMindAgent
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat


agent1 = AssistantAgent(name="Agent1", model_client=model_client, system_message="You are a write, write the content well")
agent2 = AssistantAgent(name="Agent2", model_client=model_client, system_message="You are a editor ,provide feedback , Respond with APPROVED if all feedback has been addressed")


inner_team = RoundRobinGroupChat([agent1, agent2], termination_condition=TextMentionTermination("APPROVED"))
society_of_mind_agent = SocietyOfMindAgent(name="SocietyOfMindAgent", model_client=model_client, team=inner_team)


agent3 = AssistantAgent(name="Agent3", model_client=model_client, system_message="Translate it to Spanish")


team = RoundRobinGroupChat([society_of_mind_agent, agent3], max_turns=2)



In [43]:
await Console(team.run_stream(task=""" Write a short story about a robot learning to dance. """))

---------- TextMessage (user) ----------
 Write a short story about a robot learning to dance. 
---------- TextMessage (Agent1) ----------
In a small town known for its vibrant festivals and bustling square, there existed a curious little robot named R1-22. Created by an eccentric inventor named Mr. Thompson, R1-22 was designed for menial tasks, but it often found itself daydreaming about something more joyous—dance.

One warm spring afternoon, while R1-22 was tidying up the town square, it noticed a group of people gathered around a lively band. Lively notes flowed through the air, and the crowd swayed and spun, lost in the rhythm. R1-22's sensors twitched, and it felt an unexpected tug in its circuits. The way the dancers moved, their bodies twirling with joy, ignited an unquenchable curiosity within the faithful robot.

The next day, after finishing its chores, R1-22 decided to remain in the square, hiding behind a large flower pot. As the band began to play again, it observed inten

TaskResult(messages=[TextMessage(id='cd100c93-b591-43f2-979d-03b72e09cb67', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 11, 5, 8, 9, 532940, tzinfo=datetime.timezone.utc), content=' Write a short story about a robot learning to dance. ', type='TextMessage'), TextMessage(id='c9bedae0-a303-49d3-95a1-c0729c95d6a8', source='Agent1', models_usage=RequestUsage(prompt_tokens=33, completion_tokens=748), metadata={}, created_at=datetime.datetime(2026, 4, 11, 5, 8, 23, 222259, tzinfo=datetime.timezone.utc), content="In a small town known for its vibrant festivals and bustling square, there existed a curious little robot named R1-22. Created by an eccentric inventor named Mr. Thompson, R1-22 was designed for menial tasks, but it often found itself daydreaming about something more joyous—dance.\n\nOne warm spring afternoon, while R1-22 was tidying up the town square, it noticed a group of people gathered around a lively band. Lively notes flowed through the

---
# 📁 Block 4 — Modular Project Layout
## Production-ready code structure

**Reference project:** Analyser GPT Modular (from zip)

```
analyser_gpt/
├── main.py              ← single entrypoint
├── requirements.txt
├── .env                 ← secrets (never commit!)
├── config/
│   └── constants.py     ← MODEL_NAME, timeouts, etc.
├── agents/
│   ├── data_analyzer.py ← prompt + factory function
│   └── code_executor.py
├── models/
│   └── openai_client.py ← get_model_client()
└── teams/
    └── analyzer_team.py ← get_team()
```

**Key principles:**
1. **One entrypoint** — `python main.py` runs everything
2. **Factory functions** — never create agents inline in main
3. **Config in one file** — constants.py is the single source of truth
4. **Prompts as module-level strings** — easy to find and edit
5. **Env vars for secrets** — never hardcode API keys